In [ ]:
from praatio import textgrid
import matplotlib.pyplot as plt
import soundfile as sf
from gudhi.point_cloud import timedelay
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from ripser import ripser
from persim import plot_diagrams
from scipy import sparse

import plotly.graph_objects as go
import plotly.offline as py
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff
%matplotlib qt5

inputFile="/additional/data/ALL_ENG_ENG_HT1/ALL_049_F_ENG_ENG_HT1.Textgrid"
tg=textgrid.openTextgrid(inputFile,includeEmptyIntervals=False)
phoneTier=tg.getTier('phones')

# vowels in paper: 'ə': element=2; 'ɔj': element=0
# voiced consonants in paper: 'n': element=0; 'ŋ': element=2
# voiceless consonants in paper: 't': element=4; 's': element=7

specified_phone=['s']
#vowel_phones=['ɔj','ɛ','ə','ɪ','aj','ɑ','æ','i','o','ʊ','aw','e','u','a']
#consonant_phones=['b','f','m','ɹ','ð','w','h','p','t','z','n','ɡ','dʒ','s','ʃ','v','l','ŋ','k','θ','j','tʃ','ʒ','d']

vowel_phones=[]#['ə','ɔj']
#consonant_phones=['f','p','θ','h','t','s','ʃ','k','tʃ']#voiceless_consonant
consonant_phones=[]#['t','s']#voiceless_consonant chosen one
#consonant_phones=['b','d','ɡ','v','l','m','n','ɹ','ð','z','dʒ','ŋ','j','ʒ']#voiced_consonant
consonant_phones=[]#['n','ŋ']#voiced_consonant chosen one


wavFile='/additional/data/ALL_ENG_ENG_HT1/ALL_049_F_ENG_ENG_HT1.wav'
sig,samplerate=sf.read(wavFile)
specified_phone_list=[ele for ele in phoneTier.entries if ele[2] in specified_phone]

M=100
max_edge_length=1

# wav_fraction_finder is to find the corresponding wav signal according to interval
def wav_fraction_finder(start_time, end_time):
    sig_fraction=sig[int(start_time*samplerate):int(end_time*samplerate)]
    return sig_fraction

# head_tail_scissor is to erase signal in head and tail that has amplitude smaller than 0.05
# can also use it to see if the length of renewing signal is greater than 500 or not 
def head_tail_scissor(sig):
    valid_interval=[index for index in range(len(sig)) if (sig[index]>0.03)]
    if len(valid_interval)==0:
        return False,sig
    head=min(valid_interval)
    tail=max(valid_interval)
    sig=sig[head:tail+1]
    if tail-head<500:
        return False,sig
    return True,sig

# principle_frequency_finder is to find the principle frequency of a speech signal
def principle_frequency_finder(sig):
    t=int(len(sig)/2)
    corr=np.zeros(t)

    for index in np.arange(t):
        ACF_delay=sig[index:]
        L=(t-index)/2
        m = np.sum(sig[int(t-L):int(t+L+1)]**2) + np.sum(ACF_delay[int(t-L):int(t+L+1)]**2)
        r = np.sum(sig[int(t-L):int(t+L+1)]*ACF_delay[int(t-L):int(t+L+1)])
        corr[index] = 2*r/m

    zc = np.zeros(corr.size-1)
    zc[(corr[0:-1] < 0)*(corr[1::] > 0)] = 1
    zc[(corr[0:-1] > 0)*(corr[1::] < 0)] = -1

    admiss = np.zeros(corr.size)
    admiss[0:-1] = zc
    for i in range(1, corr.size):
        if admiss[i] == 0:
            admiss[i] = admiss[i-1]

    maxes = np.zeros(corr.size)
    maxes[1:-1] = (np.sign(corr[1:-1] - corr[0:-2])==1)*(np.sign(corr[1:-1] - corr[2::])==1)
    maxidx = np.arange(corr.size)
    maxidx = maxidx[maxes == 1]
    max_index = 0
    if len(corr[maxidx]) > 0:
        max_index = maxidx[np.argmax(corr[maxidx])]

    return (max_index, corr)
    
specified_valid_list=[head_tail_scissor(wav_fraction_finder(ele[0],ele[1]))[1] for ele in specified_phone_list if head_tail_scissor(wav_fraction_finder(ele[0],ele[1]))[0]]
print('There are ',str(len(specified_valid_list)),' phones in the specified list.')


There are  28  phones in the specified list.


In [113]:
element=7
data=specified_valid_list[element]
fig=plt.figure()
plt.plot(data)

In [109]:
point_Cloud=timedelay.TimeDelayEmbedding(M, 1, 1)
Points=point_Cloud(data)
print(len(Points))
X=StandardScaler().fit_transform(Points)
pca=PCA(n_components=3,whiten=True)
X_PCA=pca.fit_transform(X)
fig = plt.figure()
ax=fig.add_subplot(1,1,1,projection='3d')
ax.plot3D(X_PCA[:,0],X_PCA[:,1],X_PCA[:,2])

1453


In [ ]:
fig = go.Figure(data=go.Scatter3d(
    x=X_PCA[:,0], y=X_PCA[:,1], z=X_PCA[:,2],
    marker=dict(
        size=4,
        color=X_PCA[:,2],
        colorscale='Viridis',#'sunset','thermal',
    ),
    line=dict(
        color='#0000b3',
        width=1
    )
))

fig.update_layout(
    width=500,
    height=500,
    template='plotly_white',
    autosize=False,
    scene=dict(
        camera=dict(
            up=dict(
                x=0,
                y=0,
                z=1
            ),
            eye=dict(
                x=1.2,
                y=1.2,
                z=1.2,
            )
        ),
        aspectratio = dict( x=1, y=1, z=0.8 ),
        aspectmode = 'manual',
    ),
)
fig.show()

In [114]:
fig1 = plt.figure()
data=specified_valid_list[element]
point_Cloud=timedelay.TimeDelayEmbedding(M, 1,1)
Points=point_Cloud(data)
print(len(Points))
dgms = ripser(Points,maxdim=1)['dgms']
plot_diagrams(dgms,lifetime=True)

1453
